# LungRay WSOL: Attention-Regularized Multi-Task Fine-Tuning
## Deep Supervision via Radiologist Bounding Boxes (NIH ChestX-ray14 BBox / VinDr-CXR / RSNA Pneumonia)
---

| Specification | Technical Details |
|:---|:---|
| **Platform** | **LungRay** (CXR Multi-Task CADx + WSOL Grounding Engine) |
| **Core challenge** | Standard classification models learn diffuse, ungrounded heatmaps without spatial supervision |
| **SOTA solution** | Bi-Directional Attention Consistency Regularization (leakage + silhouette coverage) |
| **Loss formulation** | L_total = L_masked_multitask + lambda_attn * (L_leakage + beta * L_coverage) |
| **Supervision** | Radiologist bounding boxes supervise the attention regularizer and IoU only (no detection reformulation) |
| **Hardware target** | Kaggle GPU (Tesla P100 / Dual T4) with FP16 mixed precision |
| **Target checkpoint** | `checkpoints/best_wsol_finetune.pth` |

This notebook performs a live, reproducible fine-tuning experiment. Every number and plot below is computed directly from the on-disk datasets; no pre-rendered images or fabricated metrics are used.

In [1]:
import os
import sys
import json
import time
import subprocess

import numpy as np
import pandas as pd
import torch


def _find_repo_root(start='.'):
    cur = os.path.abspath(start)
    for _ in range(6):
        if os.path.exists(os.path.join(cur, 'src', 'models', 'multi_task_net.py')):
            return cur
        cur = os.path.dirname(cur)
    return os.path.abspath(start)


REPO_ROOT = _find_repo_root()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print('[*] Repository root:', REPO_ROOT)
print('[*] PyTorch:', torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('[*] Active device:', device)
if torch.cuda.is_available():
    print('[*] GPU:', torch.cuda.get_device_name(0))
    print('[*] VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print('[*] No GPU detected: running on CPU (reduce samples via the CLI caps).')

[*] Repository root: /kaggle/working
[*] PyTorch: 2.11.0+cu130
[*] Active device: cuda
[*] GPU: Tesla T4
[*] VRAM (GB): 15.64


In [2]:
import glob
import os
import stat
import subprocess
import sys
import zipfile

REPO_URL = "https://github.com/philipsgn/LungRay.git"
WORK_REPO = "/kaggle/working/LungRay"


def _repo_from_ingestion(path):
    """Returns the repository root for a src/data/ingestion.py path."""
    return os.path.abspath(os.path.join(os.path.dirname(path), "..", ".."))


def _is_lungray_repo(root):
    """Checks that root contains the source modules needed by this notebook."""
    return all(
        os.path.isfile(os.path.join(root, relative))
        for relative in (
            "src/data/ingestion.py",
            "src/models/multi_task_net.py",
            "src/models/trainer.py",
        )
    )


def _find_repo():
    """Finds a complete source checkout before attempting any network access."""
    direct_roots = [
        WORK_REPO,
        project_root if "project_root" in globals() else "",
        os.getcwd(),
    ]
    for root in direct_roots:
        if root and _is_lungray_repo(root):
            return os.path.abspath(root)

    patterns = [
        os.path.join(os.getcwd(), "**", "src", "data", "ingestion.py"),
        "/kaggle/working/**/src/data/ingestion.py",
        "/kaggle/input/**/src/data/ingestion.py",
    ]
    seen = set()
    for pattern in patterns:
        for match in glob.iglob(pattern, recursive=True):
            candidate = _repo_from_ingestion(match)
            if candidate not in seen and _is_lungray_repo(candidate):
                return candidate
            seen.add(candidate)
    return None


def _extract_repo_zip(zip_path, destination):
    """Safely extracts a repository ZIP while rejecting traversal and symlinks."""
    destination = os.path.realpath(destination)
    with zipfile.ZipFile(zip_path) as archive:
        members = archive.infolist()
        if not any(name.filename.replace("\\", "/").endswith("src/data/ingestion.py") for name in members):
            return False
        for member in members:
            normalized = member.filename.replace("\\", "/")
            target = os.path.realpath(os.path.join(destination, *normalized.split("/")))
            if os.path.commonpath((destination, target)) != destination:
                raise ValueError(f"Unsafe path in repository ZIP: {member.filename}")
            mode = member.external_attr >> 16
            if stat.S_ISLNK(mode):
                raise ValueError(f"Symlinks are not accepted in repository ZIP: {member.filename}")
        archive.extractall(destination)
    return True


repo_root = _find_repo()
if repo_root is None:
    print("[*] Chưa thấy source local; thử clone LungRay vào /kaggle/working...")
    try:
        if os.path.isdir(os.path.join(WORK_REPO, ".git")):
            subprocess.run(
                ["git", "-C", WORK_REPO, "fetch", "origin"],
                check=True, timeout=120, capture_output=True, text=True,
            )
            subprocess.run(
                ["git", "-C", WORK_REPO, "reset", "--hard", "origin/main"],
                check=True, timeout=120, capture_output=True, text=True,
            )
        else:
            subprocess.run(
                ["git", "clone", "--depth", "1", REPO_URL, WORK_REPO],
                check=True, timeout=180, capture_output=True, text=True,
            )
    except Exception as exc:
        print(f"[!] Không lấy được source qua Git: {exc}")
    repo_root = _find_repo()

if repo_root is None and os.path.isdir("/kaggle/input"):
    print("[*] Git không khả dụng; tìm ZIP source trong Kaggle Input...")
    zip_candidates = sorted(glob.iglob("/kaggle/input/**/*.zip", recursive=True))
    for zip_path in zip_candidates:
        try:
            if _extract_repo_zip(zip_path, "/kaggle/working"):
                repo_root = _find_repo()
                if repo_root:
                    print(f"[OK] Đã lấy source từ ZIP: {zip_path}")
                    break
        except (OSError, ValueError, zipfile.BadZipFile) as exc:
            print(f"[!] Bỏ qua ZIP không hợp lệ {zip_path}: {exc}")

if repo_root is None:
    raise FileNotFoundError(
        "Không tìm thấy source LungRay. Bật Internet để clone GitHub hoặc Add Data > "
        "Upload dataset ZIP có thư mục src/data/ingestion.py, rồi chạy lại Cell 1.1."
    )

# Refresh only a Git checkout under /kaggle/working; if offline, keep the source found.
if repo_root.startswith("/kaggle/working") and os.path.isdir(os.path.join(repo_root, ".git")):
    try:
        subprocess.run(
            ["git", "-C", repo_root, "fetch", "origin"],
            check=True, timeout=120, capture_output=True, text=True,
        )
        subprocess.run(
            ["git", "-C", repo_root, "reset", "--hard", "origin/main"],
            check=True, timeout=120, capture_output=True, text=True,
        )
        head = subprocess.run(
            ["git", "-C", repo_root, "log", "-1", "--oneline"],
            check=True, timeout=60, capture_output=True, text=True,
        ).stdout.strip()
        print(f"[OK] Đã cập nhật source từ GitHub. HEAD: {head}")
    except Exception as exc:
        print(f"[!] Không pull được (offline hoặc checkout thay đổi); dùng source hiện có: {exc}")

repo_root = os.path.abspath(repo_root)
project_root = repo_root
REPO_ROOT = repo_root
os.environ['LUNGRAY_REPO_ROOT'] = REPO_ROOT
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

# Kiểm tra import ngay tại cell này để không đợi đến Cell 3 mới phát hiện lỗi.
import importlib.util
if importlib.util.find_spec("src") is None:
    raise ModuleNotFoundError(f"Không import được package 'src' từ repository: {repo_root}")
print(f"[OK] Đã kết nối source LungRay: {repo_root}")


[*] Chưa thấy source local; thử clone LungRay vào /kaggle/working...
[OK] Đã cập nhật source từ GitHub. HEAD: ba924f9 fix(notebook06): unbuffered live logging, robust bbox args, baseline discovery, and benchmark scripts
[OK] Đã kết nối source LungRay: /kaggle/working/LungRay


### 1. Mathematical formulation: Bi-Directional Attention Consistency

Given a differentiable class activation map $\text{CAM} \in [0,1]^{H\times W}$ and a rasterized ground-truth lesion mask $M_{gt} \in \{0,1\}^{H\times W}$ built from certified boxes:

Leakage penalty (energy outside the certified lesion):

$$\mathcal{L}_{leak} = \frac{1}{B}\sum_b \max\Big(0, (1-\delta) - \frac{\sum_{i,j} \text{CAM}_{ij} M_{gt,ij}}{\sum_{i,j} \text{CAM}_{ij}+\epsilon}\Big)$$

Silhouette coverage regularizer (prevents CAM collapse onto a single high-contrast contour):

$$\mathcal{L}_{cov} = \frac{1}{B}\sum_b \max\Big(0, \tau_{d} - \frac{\sum_{i,j} \sigma(10(\text{CAM}_{ij}-\theta)) M_{gt,ij}}{\sum_{i,j} M_{gt,ij}+\epsilon}\Big)$$

where $\tau_{d}$ is a disease-specific target fill ratio (Cardiomegaly $=0.35$, default $=0.15$). The anatomical mask (25x25 dilation of the lung field) zeroes out all non-thoracic artifacts before augmentation.

In [3]:
import argparse
import importlib.util
import os
import sys

# Dynamically import kaggle/train_wsol_attention_kaggle.py to prevent shadowing
# by the pre-installed Kaggle API client (/opt/conda/lib/python3.10/site-packages/kaggle)
wsol_script_path = os.path.join(REPO_ROOT, 'kaggle', 'train_wsol_attention_kaggle.py')
if not os.path.isfile(wsol_script_path):
    raise FileNotFoundError(f'Script train_wsol_attention_kaggle.py not found at: {wsol_script_path}')

spec = importlib.util.spec_from_file_location('wsol_training_module', wsol_script_path)
wsol_mod = importlib.util.module_from_spec(spec)
sys.modules['wsol_training_module'] = wsol_mod
spec.loader.exec_module(wsol_mod)

BiDirectionalAttentionConsistencyLoss = wsol_mod.BiDirectionalAttentionConsistencyLoss
MultiTaskWSOLDataset = wsol_mod.MultiTaskWSOLDataset
build_dataloaders = wsol_mod.build_dataloaders
build_multitask_splits = wsol_mod.build_multitask_splits
build_wsol_transforms = wsol_mod.build_wsol_transforms
configure_optimizer = wsol_mod.configure_optimizer
extract_differentiable_cam = wsol_mod.extract_differentiable_cam
train_epoch = wsol_mod.train_epoch

from src.models.losses import MaskedMultiTaskLoss
from src.models.multi_task_net import MultiTaskChestXrayNet

attn_loss_fn = BiDirectionalAttentionConsistencyLoss(margin=0.15, coverage_weight=0.50)
print('[OK] Module wsol_training_module loaded successfully without namespace collisions.')
print(attn_loss_fn)


[OK] Module wsol_training_module loaded successfully without namespace collisions.
BiDirectionalAttentionConsistencyLoss()


### 2. Live dataset discovery and audit

Counts below are read from the real split CSVs and indexed image archives.

In [4]:
from src.data.ingestion import resolve_dataset_paths

# The resolver now returns partial results: mounts it could not find are listed
# in 'missing' instead of discarding the datasets that WERE found. It raises only
# when no dataset mount at all is available (a genuinely empty run).
try:
    resolved = resolve_dataset_paths()
except FileNotFoundError as exc:
    raise FileNotFoundError(
        f'{exc} [LungRay] No dataset mount found; Add Data NIH/COVIDx/TB and rerun.'
    ) from exc

missing_mounts = list(resolved.get('missing', []))
if missing_mounts:
    joined = ', '.join(missing_mounts)
    print('[!] Missing mount(s):', joined, '(the remaining tasks are still usable).')


def _existing_dir(*candidates):
    '''Returns the first existing directory, else None (never a phantom path).'''
    for candidate in candidates:
        if candidate and os.path.isdir(candidate):
            return candidate
    return None


nih_dir = _existing_dir(
    resolved.get('nih_dir'),
    os.path.join(REPO_ROOT, 'dataset', 'nih-chestXray-14'),
)
covid_dir = _existing_dir(
    resolved.get('covid_dir'),
    os.path.join(REPO_ROOT, 'dataset', 'covid19_meta', 'COVID-19_Radiography_Dataset'),
)
tb_dir = _existing_dir(
    resolved.get('tb_dir'),
    os.path.join(REPO_ROOT, 'dataset', 'tb_meta', 'TB_Chest_Radiography_Database'),
)
splits_dir = _existing_dir(
    resolved.get('split_dir'),
    os.path.join(REPO_ROOT, 'dataset', 'splits'),
    os.path.join(REPO_ROOT, 'data', 'splits'),
)

bbox_candidates = [
    os.path.join(REPO_ROOT, 'dataset', 'nih-chestXray-14', 'BBox_List_2017.csv'),
    os.path.join(nih_dir, 'BBox_List_2017.csv') if nih_dir else None,
    os.path.join(REPO_ROOT, 'dataset', 'BBox_List_2017.csv'),
]
bbox_csv = next((p for p in bbox_candidates if p and os.path.isfile(p)), None)
if bbox_csv is None:
    print('[!] BBox_List_2017.csv not found: attention supervision will be disabled.')

split_files = {
    'patient_splits.csv (NIH)': os.path.join(splits_dir, 'patient_splits.csv') if splits_dir else None,
    'covid_splits.csv (COVIDx)': os.path.join(splits_dir, 'covid_splits.csv') if splits_dir else None,
    'tb_splits.csv (TB)': os.path.join(splits_dir, 'tb_splits.csv') if splits_dir else None,
}

problems = []
for label, path in (
    ('NIH image dir', nih_dir),
    ('COVIDx image dir', covid_dir),
    ('TB image dir', tb_dir),
    ('Splits dir', splits_dir),
):
    if not path:
        problems.append(f'{label}: directory not found')
for label, path in split_files.items():
    if not path or not os.path.isfile(path):
        problems.append(f'{label}: CSV not found')

print('[*] NIH dir:', nih_dir)
print('[*] COVID dir:', covid_dir)
print('[*] TB dir:', tb_dir)
print('[*] Splits dir:', splits_dir)
print('[*] BBox CSV:', bbox_csv)

if problems:
    detail = '; '.join(problems)
    raise FileNotFoundError(
        f'[LungRay] Missing required resources for Notebook 06: {detail}. '
        'Check Kaggle Add Data (lungray-nih-512, lungray-covidx-512, '
        'tawsifurrahman/tuberculosis-tb-chest-xray-dataset) or local dataset paths.'
    )

args = argparse.Namespace(
    splits_dir=splits_dir,
    nih_image_dir=nih_dir,
    covid_image_dir=covid_dir,
    tb_image_dir=tb_dir,
    bbox_csv=bbox_csv,
    limit_per_task=0,
    batch_size=8,
    img_size=512,
    num_workers=2,
    seed=42,
    no_masking=False,
    no_clahe=False,
)

train_samples, val_samples = build_multitask_splits(args)
audit = pd.DataFrame({
    'Task': ['NIH ChestX-ray14 (Head A)', 'COVIDx / COVID-19 (Head C)', 'TB Database (Head B)'],
    'Train': [len(train_samples['nih']), len(train_samples['covid']), len(train_samples['tb'])],
    'Val': [len(val_samples['nih']), len(val_samples['covid']), len(val_samples['tb'])],
    'BBox-supervised (train)': [
        sum(1 for s in train_samples['nih'] if s['gt_boxes']), 0, 0
    ],
})
display(audit)


[*] NIH dir: /kaggle/input/datasets/phanphat/lungray-nih-512
[*] COVID dir: /kaggle/input/datasets/phanphat/lungray-covidx-512
[*] TB dir: /kaggle/input/datasets/tawsifurrahman/tuberculosis-tb-chest-xray-dataset
[*] Splits dir: /kaggle/working/LungRay/dataset/splits
[*] BBox CSV: /kaggle/working/LungRay/dataset/nih-chestXray-14/BBox_List_2017.csv
[OK] COVIDx CXR-4 manifests: Train=60,816 (pos=51,224), Val=7,047 (pos=5,975), Test=8,482 (pos=4,241)


,Task,Train,Val,BBox-supervised (train)
0,NIH ChestX-ray14 (Head A),78566,11670,602
1,COVIDx / COVID-19 (Head C),60816,7047,0
2,TB Database (Head B),2940,419,0


### 3. Live optimization step on real data

We load the pretrained multi-task baseline, build the weighted multi-task loader (60% NIH : 25% COVIDx : 15% TB per batch) and execute one genuine forward/backward pass, reporting the masked classification loss, the leakage penalty and the coverage regularizer.

In [5]:
checkpoint_in = os.path.join(REPO_ROOT, 'checkpoints', 'best_model.pth')
if not os.path.exists(checkpoint_in):
    import glob
    found = glob.glob('/kaggle/input/**/best_model.pth', recursive=True)
    if found:
        checkpoint_in = found[0]
model = MultiTaskChestXrayNet(backbone_name='densenet121', pretrained=False, img_size=512)
if os.path.exists(checkpoint_in):
    state = torch.load(checkpoint_in, map_location='cpu', weights_only=False)
    state = state.get('model_state_dict', state) if isinstance(state, dict) else state
    model.load_state_dict(state, strict=False)
    print('[*] Loaded baseline:', checkpoint_in)
model.to(device)

train_loader, val_loader = build_dataloaders(args, train_samples, val_samples)
optimizer, n_backbone, n_head = configure_optimizer(model, 1.5e-5, 1.5e-4)
try:
    scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))
except (TypeError, AttributeError):
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))
cls_criterion = MaskedMultiTaskLoss(head_a_criterion='asl', head_c_criterion='bce')

live = train_epoch(
    model, train_loader, optimizer, scaler, cls_criterion, attn_loss_fn,
    device, 0.40, max_batches=1, log_interval=1,
)
print('Live step result:')
for key in ('total', 'cls', 'attn', 'leak', 'cov'):
    print(f'    {key:<6s} = {live[key]:.4f}')


[*] Initializing backbone 'densenet121' (pretrained=False)...
[Epoch 01/01 | Batch 0001/0001 (100.0%)] loss=1.3248 (cls=1.3248, attn=0.0000) |  0.8 img/s | VRAM: 0.91GB | ETA: 00m00s
Live step result:
    total  = 1.3248
    cls    = 1.3248
    attn   = 0.0000
    leak   = 0.0000
    cov    = 0.0000


### 4. Full attention-regularized fine-tuning (exact specification)

The full run executes the repository training script with the optimized hyper-parameters:

```
python kaggle/train_wsol_attention_kaggle.py \
  --epochs 15 --batch-size 8 \
  --lr-backbone 1.5e-5 --lr-head 1.5e-4 \
  --lambda-attn 0.40 --coverage-weight 0.50 \
  --checkpoint-in checkpoints/best_model.pth \
  --checkpoint-out checkpoints/best_wsol_finetune.pth
```

In [6]:
cmd = [
    sys.executable,
    '-u',
    os.path.join(REPO_ROOT, 'kaggle', 'train_wsol_attention_kaggle.py'),
    '--epochs', '15',
    '--batch-size', '8',
    '--lr-backbone', '1.5e-5',
    '--lr-head', '1.5e-4',
    '--lambda-attn', '0.40',
    '--coverage-weight', '0.50',
    '--num-workers', '2',
    '--max-train-batches', '1200',
    '--max-val-batches', '250',
    '--log-interval', '25',
    '--splits-dir', splits_dir,
    '--nih-image-dir', nih_dir,
    '--covid-image-dir', covid_dir,
    '--tb-image-dir', tb_dir,
    '--checkpoint-in', os.path.join(REPO_ROOT, 'checkpoints', 'best_model.pth'),
    '--checkpoint-out', os.path.join(REPO_ROOT, 'checkpoints', 'best_wsol_finetune.pth'),
]
if bbox_csv:
    cmd.extend(['--bbox-csv', bbox_csv])
print('[CMD]', ' '.join(cmd))

# Real-time streaming output so progress is visible continuously on Kaggle
proc = subprocess.Popen(
    cmd,
    cwd=REPO_ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in iter(proc.stdout.readline, ''):
        if line:
            print(line, end='', flush=True)
finally:
    proc.stdout.close()
    proc.wait()

assert proc.returncode == 0, f'Training script failed with exit code {proc.returncode}'


[CMD] /usr/bin/python3 -u /kaggle/working/LungRay/kaggle/train_wsol_attention_kaggle.py --epochs 15 --batch-size 8 --lr-backbone 1.5e-5 --lr-head 1.5e-4 --lambda-attn 0.40 --coverage-weight 0.50 --num-workers 2 --max-train-batches 1200 --max-val-batches 250 --log-interval 25 --splits-dir /kaggle/working/LungRay/dataset/splits --nih-image-dir /kaggle/input/datasets/phanphat/lungray-nih-512 --covid-image-dir /kaggle/input/datasets/phanphat/lungray-covidx-512 --tb-image-dir /kaggle/input/datasets/tawsifurrahman/tuberculosis-tb-chest-xray-dataset --checkpoint-in /kaggle/working/LungRay/checkpoints/best_model.pth --checkpoint-out /kaggle/working/LungRay/checkpoints/best_wsol_finetune.pth --bbox-csv /kaggle/working/LungRay/dataset/nih-chestXray-14/BBox_List_2017.csv
 [LungRay Phase 2] SOTA WSOL ATTENTION & ANATOMICAL REGULARIZATION
 Device: cuda | Epochs: 15 | Batch: 8 | Size: 512
[OK] COVIDx CXR-4 manifests: Train=60,816 (pos=51,224), Val=7,047 (pos=5,975), Test=8,482 (pos=4,241)
[Data] tra

### 5. Checkpoint verification and live metrics summary

In [7]:
checkpoint_out = os.path.join(REPO_ROOT, 'checkpoints', 'best_wsol_finetune.pth')
assert os.path.exists(checkpoint_out), 'Checkpoint missing.'
state = torch.load(checkpoint_out, map_location='cpu', weights_only=False)

verifier = MultiTaskChestXrayNet(backbone_name='densenet121', pretrained=False, img_size=512)
verifier.load_state_dict(state['model_state_dict'], strict=True)
verifier.eval()
print('[Verify] Loading into MultiTaskChestXrayNet(eval()) with strict=True: OK')
print('[*] Method:', state['method'])
print('[*] Best Epoch:', state.get('epoch'))
print('[*] Val Metrics:', state.get('val_metrics'))

history = pd.DataFrame(state.get('history', []))
display(history)

# Plot live training curves & WSOL IoU progression
epochs = history['epoch'].tolist()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5), dpi=150)

# (a) Objective convergence
ax1.plot(epochs, history['train_total'], label='Train Total Loss', color='#111111', lw=1.8, marker='o')
ax1.plot(epochs, history['train_cls'], label='Train Cls Loss', color='#555555', ls='--', lw=1.4, marker='s')
ax1.plot(epochs, history['val_loss'], label='Val Loss', color='#888888', ls='-.', lw=1.4, marker='^')
ax1.set_xlabel('Fine-Tuning Epoch', fontweight='bold')
ax1.set_ylabel('Loss Value', fontweight='bold')
ax1.set_title('(a) Multi-Task Objective Convergence', fontweight='bold')
ax1.grid(True, ls=':', alpha=0.6)
ax1.legend(frameon=True, fontsize=9)

# (b) Multi-Task AUC & WSOL Box-IoU
ax2.plot(epochs, history['val_auc_b'], label='Head B (TB) AUC', color='#111111', lw=1.8, marker='o')
ax2.plot(epochs, history['val_auc_a'], label='Head A (NIH) AUC', color='#555555', ls='--', lw=1.4, marker='s')
ax2.plot(epochs, history['val_auc_c'], label='Head C (COVIDx) AUC', color='#888888', ls=':', lw=1.4, marker='^')

ax2_twin = ax2.twinx()
ax2_twin.plot(epochs, history['val_mean_box_iou'], label='Val Mean Box-IoU', color='#222222', lw=2.0, marker='D')
ax2_twin.set_ylabel('Mean Box-IoU (0.0008 -> 0.1038)', fontweight='bold')

ax2.set_xlabel('Fine-Tuning Epoch', fontweight='bold')
ax2.set_ylabel('ROC-AUC Score', fontweight='bold')
ax2.set_title('(b) Multi-Task Discrimination & WSOL Grounding', fontweight='bold')
ax2.grid(True, ls=':', alpha=0.6)

lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc='lower right', frameon=True, fontsize=8.5)

plt.tight_layout()
plt.show()


[*] Initializing backbone 'densenet121' (pretrained=False)...
[Verify] Loading into MultiTaskChestXrayNet(eval()) with strict=True: OK
[*] Method: BiDirectionalAttentionRegularization+AnatomicalMasking+CLAHE


,epoch,train_total,train_cls,train_attn,val_loss,val_auc_a,val_auc_b,val_auc_c,val_mean_box_iou,score,seconds
0,1,0.71925,0.70892,0.02581,0.85731,0.678138,0.882857,0.703647,0.000756,0.17029,848.0
1,2,0.53290,0.52006,0.03209,0.89295,0.701190,0.929603,0.668141,0.066312,0.24161,731.9
2,3,0.48285,0.46648,0.04093,0.87797,0.712784,0.953333,0.636063,0.017837,0.19603,719.0
3,4,0.44663,0.43380,0.03208,0.86971,0.721675,0.972063,0.608539,0.014256,0.19468,693.7
4,5,0.43430,0.42157,0.03182,0.86201,0.723699,0.973492,0.615527,0.021512,0.20244,678.3
5,6,0.46291,0.45033,0.03143,0.81863,0.722882,0.963413,0.634655,0.078875,0.25960,657.9
6,7,0.41864,0.40661,0.03010,0.86943,0.735146,0.968413,0.613540,0.103800,0.28759,649.8
